In [1]:
!nvidia-smi

Fri Nov 20 20:36:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [18]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# if torch.cuda.is_available():
#     print("GPU:", torch.cuda.get_device_name(cpu))
#     print("GPU memory:", round(
#         torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
#     ), "GB")
# else:
#     raise RuntimeError("No GPU detected. Enable GPU runtime first.")

PyTorch: 2.14.1+cpu
CUDA available: False


In [7]:
!pip install -q ultralytics==8.4.215

In [8]:
import ultralytics
import torch

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics: 8.4.215
PyTorch: 2.14.1+cpu
CUDA: False


In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
import zipfile
from pathlib import Path
import shutil

ZIP_PATH = Path(
    "/content/drive/MyDrive/PlantCareAI_GPU/input/ip102_yolo_fixed.zip"
)

EXTRACT_DIR = Path("/content/ip102")

assert ZIP_PATH.exists(), "Corrected ZIP not found!"

# Remove only the incomplete extracted dataset
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)

EXTRACT_DIR.mkdir(parents=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

DATASET_DIR = EXTRACT_DIR / "yolo"

for split in ["train", "val", "test"]:
    image_dir = DATASET_DIR / "images" / split
    label_dir = DATASET_DIR / "labels" / split

    image_count = len(list(image_dir.glob("*.jpg")))
    label_count = len(list(label_dir.glob("*.txt")))

    print(f"\n{split.upper()}")
    print("Images:", image_count)
    print("Labels:", label_count)

    assert image_count == label_count, "Image/label count mismatch!"


TRAIN
Images: 13659
Labels: 13659

VAL
Images: 1518
Labels: 1518

TEST
Images: 3798
Labels: 3798


In [13]:
import re
import yaml

ORIGINAL_YAML = DATASET_DIR / "data.yaml"
COLAB_YAML = DATASET_DIR / "data_colab.yaml"

yaml_text = ORIGINAL_YAML.read_text(encoding="utf-8")

yaml_text, replacements = re.subn(
    r"(?m)^path\s*:.*$",
    f"path: {DATASET_DIR.as_posix()}",
    yaml_text,
    count=1
)

assert replacements == 1, "Could not update dataset path"

COLAB_YAML.write_text(yaml_text, encoding="utf-8")

with open(COLAB_YAML, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

print("Dataset path:", config["path"])
print("Number of classes:", len(config["names"]))
print("Class ID range:", min(config["names"]), "-", max(config["names"]))

assert len(config["names"]) == 102
assert min(config["names"]) == 0
assert max(config["names"]) == 101

print("\nYAML verification passed!")

Dataset path: /content/ip102/yolo
Number of classes: 102
Class ID range: 0 - 101

YAML verification passed!


In [14]:
from pathlib import Path

BEST_PT = Path(
    "/content/drive/MyDrive/PlantCareAI_GPU/input/yolo26n.pt"
)

assert BEST_PT.exists(), "yolo26n.pt checkpoint not found!"

print("Checkpoint found:", BEST_PT)
print("Checkpoint size:", round(BEST_PT.stat().st_size / (1024**2), 2), "MB")

Checkpoint found: /content/drive/MyDrive/PlantCareAI_GPU/input/best.pt
Checkpoint size: 6.21 MB


In [15]:
from ultralytics import YOLO
from pathlib import Path

BEST_PT = Path(
    "/content/drive/MyDrive/PlantCareAI_GPU/input/yolo26n.pt"
)

assert BEST_PT.exists(), "Checkpoint not found!"

model = YOLO(str(BEST_PT))

print("Model loaded successfully!")
print("Checkpoint:", BEST_PT)

Model loaded successfully!
Checkpoint: /content/drive/MyDrive/PlantCareAI_GPU/input/best.pt


In [22]:
baseline_model = YOLO(str(BEST_PT))

baseline_metrics = baseline_model.val(
    data=str(COLAB_YAML),
    split="val",
    imgsz=640,
    batch=16,
    device='cpu',
    workers=2,
    plots=True,
    project="/content/drive/MyDrive/PlantCareAI_GPU/runs/pest",
    name="baseline_best_640",
)

print("\nBASELINE RESULTS AT 640")
print("Precision:", baseline_metrics.box.mp)
print("Recall:", baseline_metrics.box.mr)
print("mAP50:", baseline_metrics.box.map50)
print("mAP50-95:", baseline_metrics.box.map)

Ultralytics 8.4.215 🚀 Python-3.13.18 torch-2.14.1+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 73 layers, 3,157,250 parameters, 0 gradients, 8.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 749.5±423.1 MB/s, size: 27.4 KB)
val: Scanning /content/ip102/yolo/labels/val.cache... 1518 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1518/1518 318.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 16% ━━────────── 16/95 4.4s/it 45.3s<5:46


KeyboardInterrupt: 

In [21]:
from ultralytics import YOLO

model_416 = YOLO(str(BEST_PT))

metrics_416 = model_416.val(
    data=str(COLAB_YAML),
    split="val",
    imgsz=416,
    batch=16,
    device='cpu',
    workers=2,
    plots=True,
    project="/content/drive/MyDrive/PlantCareAI_GPU/runs/pest",
    name="baseline_best_416_recheck",
)

print("\nBASELINE RECHECK AT 416")
print("Precision:", metrics_416.box.mp)
print("Recall:", metrics_416.box.mr)
print("mAP50:", metrics_416.box.map50)
print("mAP50-95:", metrics_416.box.map)

Ultralytics 8.4.215 🚀 Python-3.13.18 torch-2.14.1+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 73 layers, 3,157,250 parameters, 0 gradients, 8.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 532.6±399.5 MB/s, size: 29.1 KB)
val: Scanning /content/ip102/yolo/labels/val.cache... 1518 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1518/1518 254.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 95/95 1.4s/it 2:16
                   all       1518       1873      0.513      0.342      0.348      0.218
      rice leaf roller         14         15      0.384      0.867      0.802      0.575
 rice leaf caterpillar          7          7      0.233      0.286      0.146       0.12
     paddy stem maggot          2          2          1          0     0.0545     0.0225
    asiatic rice borer          8          9      0.216      0.333       0.13     0.0645
     yellow rice borer          9         1

**Experiment A — YOLOv8n fine-tuning at 416**

In [23]:
from pathlib import Path

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/PlantCareAI_GPU/runs/pest"
)

RUN_NAME = "ip102_yolov8n_finetune_416"

assert BEST_PT.exists()
assert COLAB_YAML.exists()

RUN_DIR = OUTPUT_DIR / RUN_NAME

assert not RUN_DIR.exists(), (
    "Run directory already exists. Choose a new run name."
)

print("Checkpoint:", BEST_PT)
print("Dataset:", COLAB_YAML)
print("Output:", RUN_DIR)
print("Ready to train!")

AssertionError: Run directory already exists. Choose a new run name.

In [30]:
#if cuda oom, retry with batch 16 on new run

import torch
from ultralytics import YOLO

assert torch.cuda.is_available()

model = YOLO(str(BEST_PT))

results = model.train(
    data=str(COLAB_YAML),

    # Training
    epochs=40,
    imgsz=416,
    batch=32,
    device=0,

    # Optimisation
    optimizer="AdamW",
    lr0=0.0001,
    lrf=0.01,
    momentum=0.9,
    weight_decay=0.0005,

    # Training control
    patience=10,
    warmup_epochs=2,
    seed=42,
    close_mosaic=10,

    # GPU and data loading
    workers=2,
    cache=False,
    amp=True,

    # Saving and validation
    save=True,
    save_period=5,
    val=True,
    plots=True,

    # Output
    project=str(OUTPUT_DIR),
    name=RUN_NAME,
    exist_ok=False,
    resume=False,
    verbose=True
)

print("\nFine-tuning completed!")

New https://pypi.org/project/ultralytics/8.4.216 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/ip102/yolo/data_colab.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=None, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/d

In [4]:
import pandas as pd
from pathlib import Path

RUN_DIR = Path(
    "/content/drive/MyDrive/PlantCareAI_GPU/runs/pest/"
    "ip102_yolov8n_finetune_416"
)

RESULTS_CSV = RUN_DIR / "results.csv"
BEST_PT = RUN_DIR / "weights" / "yolo26n.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"

assert RESULTS_CSV.exists(), "results.csv not found"
assert BEST_PT.exists(), "yolo26n.pt not found"

df = pd.read_csv(RESULTS_CSV)
df.columns = df.columns.str.strip()

print("Total epochs recorded:", len(df))
print("Best checkpoint:", BEST_PT)
print("\nAvailable columns:")
print(df.columns.tolist())

metric_columns = [
    "metrics/precision(B)",
    "metrics/recall(B)",
    "metrics/mAP50(B)",
    "metrics/mAP50-95(B)"
]

print("\nTop 5 epochs by mAP50-95:")
print(
    df.nlargest(5, "metrics/mAP50-95(B)")[
        ["epoch"] + metric_columns
    ].to_string(index=False)
)

print("\nBest checkpoint exists:", BEST_PT.exists())
print("Last checkpoint exists:", LAST_PT.exists())

Total epochs recorded: 40
Best checkpoint: /content/drive/MyDrive/PlantCareAI_GPU/runs/pest/ip102_yolov8n_finetune_416/weights/best.pt

Available columns:
['epoch', 'time', 'train/box_loss', 'train/cls_loss', 'train/dfl_loss', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'val/box_loss', 'val/cls_loss', 'val/dfl_loss', 'lr/pg0', 'lr/pg1', 'lr/pg2']

Top 5 epochs by mAP50-95:
 epoch  metrics/precision(B)  metrics/recall(B)  metrics/mAP50(B)  metrics/mAP50-95(B)
    35               0.48727            0.56085           0.54533              0.34614
    37               0.50010            0.54950           0.54401              0.34434
    38               0.50775            0.52776           0.54148              0.34309
    39               0.49211            0.55976           0.53962              0.34129
    34               0.50436            0.52741           0.53946              0.34118

Best checkpoint exists: True
Last checkpoint exists: True

In [24]:
from pathlib import Path
from ultralytics import YOLO

BEST_PT = Path(
    "/content/drive/MyDrive/PlantCareAI_GPU/runs/pest/"
    "ip102_yolov8n_finetune_416/weights/yolo26n.pt"
)

COLAB_YAML = Path("/content/ip102/yolo/data_colab.yaml")

TEST_RUN_DIR = (
    "/content/drive/MyDrive/PlantCareAI_GPU/runs/pest"
)

assert BEST_PT.exists(), "Best checkpoint not found"
assert COLAB_YAML.exists(), "Colab dataset YAML not found"

# Load the best validation checkpoint
model = YOLO(str(BEST_PT))

# Evaluate once on the held-out test split
test_results = model.val(
    data=str(COLAB_YAML),
    split="test",
    imgsz=416,
    batch=32,
    device='cpu',
    workers=2,
    plots=True,
    save_json=True,
    project=TEST_RUN_DIR,
    name="ip102_yolov8n_test_416",
    exist_ok=False,
    verbose=True
)

print("\nTEST SET EVALUATION COMPLETED")
print("Precision:", test_results.box.mp)
print("Recall:", test_results.box.mr)
print("mAP50:", test_results.box.map50)
print("mAP50-95:", test_results.box.map)

Ultralytics 8.4.215 🚀 Python-3.13.18 torch-2.14.1+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 73 layers, 3,157,250 parameters, 0 gradients, 8.8 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 5.8±2.6 MB/s, size: 43.6 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/ip102/yolo/labels/test... 3798 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3798/3798 578.8it/s 6.6s
val: New cache created: /content/ip102/yolo/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 119/119 3.0s/it 5:52
                   all       3798       4444      0.521      0.545      0.537      0.342
      rice leaf roller         35         39      0.785      0.872      0.882      0.618
 rice leaf caterpillar         23         26      0.316      0.269      0.305      0.193
     paddy s